# Transformación de IMFs a Grafo mediante Visibility Graph (HVG)

Este notebook carga los datos de las IMFs del MSCI World para su posterior transformación a grafo usando el algoritmo Horizontal Visibility Graph (HVG).


In [1]:
import pandas as pd
import numpy as np
import torch
from ts2vg import HorizontalVG
from torch_geometric.data import Data


/home/agusriscos/proyectos/ARPTools/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cargar datos de IMFs
archivo_imfs = "data/16dic25/msci_world_imfs.parquet"

df_imfs = pd.read_parquet(archivo_imfs, engine="pyarrow")
print(f"Datos cargados desde: {archivo_imfs}")
print(f"Shape del DataFrame: {df_imfs.shape}")
print(f"\nColumnas disponibles: {list(df_imfs.columns)}")
print(f"\nRango de fechas disponible:")
print(f"  - Desde: {df_imfs.index.min()}")
print(f"  - Hasta: {df_imfs.index.max()}")


Datos cargados desde: data/16dic25/msci_world_imfs.parquet
Shape del DataFrame: (3490, 11)

Columnas disponibles: ['IMF_1', 'IMF_2', 'IMF_3', 'IMF_4', 'IMF_5', 'IMF_6', 'IMF_7', 'IMF_8', 'IMF_9', 'IMF_10', 'Residuo']

Rango de fechas disponible:
  - Desde: 2012-01-12 00:00:00-05:00
  - Hasta: 2025-11-26 00:00:00-05:00


In [3]:
# Seleccionar IMF_1
imf_1 = df_imfs['IMF_1'].values
print(f"Shape de IMF_1: {imf_1.shape}")
print(f"Primeros 10 valores de IMF_1: {imf_1[:10]}")


Shape de IMF_1: (3490,)
Primeros 10 valores de IMF_1: [-0.27589986 -0.2934861   0.08549549 -1.16096792  1.30101302 -0.84105598
  1.0079569  -0.56710679 -0.46208816  0.42473901]


In [4]:
# Construir el grafo HVG (Horizontal Visibility Graph)
hvg = HorizontalVG(directed="left_to_right")
grafo_hvg = hvg.build(imf_1)

# Obtener nodos y enlaces
nodos = np.arange(len(imf_1))  # Los nodos son los índices temporales
enlaces = np.array(grafo_hvg.edges)  # Los enlaces son las conexiones de visibilidad

print(f"Número de nodos: {len(nodos)}")
print(f"Número de enlaces: {len(enlaces)}")
print(f"\nPrimeros 10 nodos: {nodos[:10]}")
print(f"\nPrimeros 10 enlaces:")
print(enlaces[:10])
print(f"\nForma de los enlaces: {enlaces.shape}")


Número de nodos: 3490
Número de enlaces: 6968

Primeros 10 nodos: [0 1 2 3 4 5 6 7 8 9]

Primeros 10 enlaces:
[[3328 3329]
 [3327 3329]
 [3325 3329]
 [3324 3329]
 [3304 3329]
 [2683 3329]
 [2054 3329]
 [2051 3329]
 [3329 3330]
 [3329 3331]]

Forma de los enlaces: (6968, 2)


In [5]:
# Convertir el grafo HVG a un objeto Data de PyTorch Geometric
# Convertir enlaces a edge_index (formato [2, num_edges])
edge_index = torch.tensor(enlaces.T, dtype=torch.long)

# Crear features de nodos usando los valores de la serie temporal
# Cada nodo tiene como feature su valor en la serie temporal
node_features = torch.tensor(imf_1, dtype=torch.float).unsqueeze(1)

# Crear el objeto Data
data = Data(x=node_features, edge_index=edge_index)

print(f"Objeto Data creado:")
print(f"  - Número de nodos: {data.num_nodes}")
print(f"  - Número de enlaces: {data.num_edges}")
print(f"  - Features de nodos shape: {data.x.shape}")
print(f"  - Edge index shape: {data.edge_index.shape}")
print(f"\nPrimeros 5 features de nodos:")
print(data.x[:5])
print(f"\nPrimeros 10 enlaces (edge_index):")
print(data.edge_index[:, :10])


Objeto Data creado:
  - Número de nodos: 3490
  - Número de enlaces: 6968
  - Features de nodos shape: torch.Size([3490, 1])
  - Edge index shape: torch.Size([2, 6968])

Primeros 5 features de nodos:
tensor([[-0.2759],
        [-0.2935],
        [ 0.0855],
        [-1.1610],
        [ 1.3010]])

Primeros 10 enlaces (edge_index):
tensor([[3328, 3327, 3325, 3324, 3304, 2683, 2054, 2051, 3329, 3329],
        [3329, 3329, 3329, 3329, 3329, 3329, 3329, 3329, 3330, 3331]])


In [6]:
# Filtrar el grafo para el rango 2020-2025 y visualizar con Plotly
import plotly.graph_objects as go
from datetime import datetime

# Obtener una copia del índice para trabajar con él
indice_original = df_imfs.index.copy()

# Verificar y convertir el índice a datetime si es necesario
if not isinstance(indice_original, pd.DatetimeIndex):
    print("Convirtiendo índice a DatetimeIndex...")
    indice_original = pd.to_datetime(indice_original)

# Normalizar timezone si existe (convertir a naive datetime)
try:
    if hasattr(indice_original, 'tz') and indice_original.tz is not None:
        print("Normalizando timezone del índice...")
        # Convertir a UTC y luego quitar timezone
        indice_original = indice_original.tz_convert('UTC').tz_localize(None)
except Exception as e:
    print(f"Advertencia al normalizar timezone: {e}")
    # Intentar método alternativo
    try:
        indice_original = pd.to_datetime(indice_original).tz_localize(None)
    except:
        pass

# Definir rango de fechas (sin timezone para compatibilidad)
fecha_inicio = pd.Timestamp('2020-01-01')
fecha_fin = pd.Timestamp('2025-12-31')

# Normalizar fechas de búsqueda también si es necesario
if hasattr(fecha_inicio, 'tz') and fecha_inicio.tz is not None:
    fecha_inicio = fecha_inicio.tz_localize(None)
if hasattr(fecha_fin, 'tz') and fecha_fin.tz is not None:
    fecha_fin = fecha_fin.tz_localize(None)

# Obtener máscara de fechas en el rango
mascara_fechas = (indice_original >= fecha_inicio) & (indice_original <= fecha_fin)
indices_filtrados = np.where(mascara_fechas)[0]

print(f"Filtrando grafo para el rango {fecha_inicio.date()} a {fecha_fin.date()}")
print(f"Nodos en el rango: {len(indices_filtrados)}")
if len(indices_filtrados) > 0:
    print(f"Índices: {indices_filtrados[0]} a {indices_filtrados[-1]}")
else:
    print("⚠ No se encontraron nodos en el rango especificado")
    print(f"Rango disponible: {indice_original.min()} a {indice_original.max()}")

# Crear un conjunto de nodos válidos para el filtro
nodos_validos = set(indices_filtrados)

# Filtrar edge_index para mantener solo enlaces entre nodos válidos
edge_index_filtrado = data.edge_index.cpu().numpy()
enlaces_filtrados = []
for i in range(edge_index_filtrado.shape[1]):
    src = int(edge_index_filtrado[0, i])
    dst = int(edge_index_filtrado[1, i])
    if src in nodos_validos and dst in nodos_validos:
        enlaces_filtrados.append([src, dst])

if len(enlaces_filtrados) > 0:
    enlaces_filtrados = np.array(enlaces_filtrados, dtype=np.int64)
else:
    enlaces_filtrados = np.array([], dtype=np.int64).reshape(0, 2)
print(f"Enlaces en el subgrafo: {len(enlaces_filtrados)}")

# Crear mapeo de índices originales a índices locales (0, 1, 2, ...)
mapeo_indices = {int(idx_original): int(idx_local) for idx_local, idx_original in enumerate(indices_filtrados)}
if len(enlaces_filtrados) > 0:
    enlaces_mapeados = np.array([[mapeo_indices[int(src)], mapeo_indices[int(dst)]] 
                                  for src, dst in enlaces_filtrados], dtype=np.int64)
else:
    enlaces_mapeados = np.array([], dtype=np.int64).reshape(0, 2)

# Obtener valores de IMF_1 y fechas para los nodos filtrados
valores_imf_filtrados = imf_1[indices_filtrados]
fechas_filtradas = indice_original[indices_filtrados]

print(f"\nDatos del subgrafo:")
print(f"  - Nodos: {len(indices_filtrados)}")
print(f"  - Enlaces: {len(enlaces_mapeados)}")
if len(fechas_filtradas) > 0:
    fecha_min = fechas_filtradas[0]
    fecha_max = fechas_filtradas[-1]
    if isinstance(fecha_min, pd.Timestamp):
        print(f"  - Rango de fechas: {fecha_min.date()} a {fecha_max.date()}")
    else:
        print(f"  - Rango de fechas: {fecha_min} a {fecha_max}")


Normalizando timezone del índice...
Filtrando grafo para el rango 2020-01-01 a 2025-12-31
Nodos en el rango: 1485
Índices: 2005 a 3489
Enlaces en el subgrafo: 2959

Datos del subgrafo:
  - Nodos: 1485
  - Enlaces: 2959
  - Rango de fechas: 2020-01-02 a 2025-11-26


In [7]:
# Crear visualización del grafo con Plotly usando layout de fuerza
# Incluyendo también la serie temporal de la IMF en la misma figura
import networkx as nx
from plotly.subplots import make_subplots

# Validar que hay datos para visualizar
if len(indices_filtrados) == 0:
    raise ValueError("No hay nodos en el rango de fechas especificado. Ajusta el rango de fechas.")

if len(enlaces_mapeados) == 0:
    print("⚠ Advertencia: No hay enlaces en el subgrafo filtrado.")

# Convertir valores a arrays de numpy para asegurar compatibilidad
valores_imf_array = np.array(valores_imf_filtrados, dtype=np.float64)

# Crear grafo NetworkX para calcular layout
print("Creando grafo NetworkX...")
G = nx.Graph()
G.add_nodes_from(range(len(indices_filtrados)))
if len(enlaces_mapeados) > 0:
    # Asegurar que los enlaces sean tuplas de enteros
    enlaces_tuplas = [(int(edge[0]), int(edge[1])) for edge in enlaces_mapeados]
    G.add_edges_from(enlaces_tuplas)
    print(f"Grafo creado con {G.number_of_nodes()} nodos y {G.number_of_edges()} enlaces")

# Calcular layout usando spring layout (fuerza dirigida)
print("Calculando layout del grafo...")
pos = nx.spring_layout(G, k=1, iterations=50, seed=42)

# Extraer posiciones
x_pos = np.array([pos[i][0] for i in range(len(indices_filtrados))])
y_pos = np.array([pos[i][1] for i in range(len(indices_filtrados))])

# Preparar texto de hover de forma más eficiente
def formatear_fecha(fecha):
    """Formatea una fecha de forma segura."""
    try:
        if isinstance(fecha, pd.Timestamp):
            return fecha.strftime('%Y-%m-%d')
        elif isinstance(fecha, str):
            return fecha
        else:
            return str(pd.Timestamp(fecha).strftime('%Y-%m-%d'))
    except:
        return str(fecha)

textos_hover = [
    f"Fecha: {formatear_fecha(fecha)}<br>Valor IMF_1: {val:.4f}<br>Índice: {idx}"
    for fecha, val, idx in zip(fechas_filtradas, valores_imf_array, indices_filtrados)
]

# Crear trazas para los enlaces de forma más eficiente
print("Creando trazas de enlaces...")
edge_x = []
edge_y = []
if len(enlaces_mapeados) > 0:
    for edge in enlaces_mapeados:
        src_idx = int(edge[0])
        dst_idx = int(edge[1])
        x0, y0 = pos[src_idx]
        x1, y1 = pos[dst_idx]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

# Crear traza de enlaces
edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    showlegend=False
)

# Calcular rango de valores para la escala de colores (compartida entre grafo y serie temporal)
valor_min = np.min(valores_imf_array)
valor_max = np.max(valores_imf_array)

# Crear traza para los nodos
print("Creando traza de nodos...")
node_trace = go.Scatter(
    x=x_pos,
    y=y_pos,
    mode='markers',
    marker=dict(
        size=8,
        color=valores_imf_array,
        colorscale='Viridis',
        cmin=valor_min,  # Rango mínimo para la escala
        cmax=valor_max,  # Rango máximo para la escala
        showscale=False,  # Ocultar colorbar del grafo, se mostrará solo en la serie temporal
        line=dict(width=1, color='black')
    ),
    text=textos_hover,
    hoverinfo='text',
    showlegend=False
)

# Crear traza para la serie temporal con colores basados en valores (mismo heatmap que el grafo)
print("Creando traza de serie temporal...")
serie_temporal_trace = go.Scatter(
    x=fechas_filtradas,
    y=valores_imf_array,
    mode='lines+markers',
    name='IMF_1',
    line=dict(color='rgba(128,128,128,0.3)', width=1),  # Línea sutil gris para conectar puntos
    marker=dict(
        size=4,
        color=valores_imf_array,
        colorscale='Viridis',
        cmin=valor_min,  # Mismo rango que el grafo
        cmax=valor_max,  # Mismo rango que el grafo
        showscale=True,  # Solo esta traza mostrará la colorbar
        colorbar=dict(title="Valor IMF_1", x=1.02, len=0.4, y=0.25),
        line=dict(width=0.5, color='white')
    ),
    hovertemplate='Fecha: %{x}<br>Valor IMF_1: %{y:.4f}<extra></extra>'
)

# Crear figura con subplots: grafo arriba, serie temporal abajo
print("Creando figura con subplots...")
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.6, 0.4],  # 60% para el grafo, 40% para la serie temporal
    subplot_titles=(
        f'Grafo HVG - MSCI World IMF_1 ({fecha_inicio.date()} a {fecha_fin.date()})',
        'Serie Temporal IMF_1'
    ),
    vertical_spacing=0.12
)

# Añadir trazas del grafo al primer subplot
fig.add_trace(edge_trace, row=1, col=1)
fig.add_trace(node_trace, row=1, col=1)

# Añadir traza de serie temporal al segundo subplot
fig.add_trace(serie_temporal_trace, row=2, col=1)

# Actualizar layout del primer subplot (grafo)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    row=1, col=1
)
fig.update_yaxes(
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    row=1, col=1
)

# Actualizar layout del segundo subplot (serie temporal)
fig.update_xaxes(
    title_text="Fecha",
    row=2, col=1
)
fig.update_yaxes(
    title_text="Valor IMF_1",
    row=2, col=1
)

# Actualizar layout general
fig.update_layout(
    title=dict(
        text=f'Grafo HVG y Serie Temporal - MSCI World IMF_1 ({fecha_inicio.date()} a {fecha_fin.date()})',
        x=0.5,
        xanchor='center',
        font=dict(size=18)
    ),
    showlegend=False,
    hovermode='closest',
    height=900,
    margin=dict(b=50, l=50, r=50, t=80),
    annotations=[
        dict(
            text=f"Nodos: {len(indices_filtrados)} | Enlaces: {len(enlaces_mapeados)}",
            showarrow=False,
            xref="paper", yref="paper",
            x=0.005, y=0.48,
            xanchor='left', yanchor='bottom',
            font=dict(size=12)
        )
    ],
    plot_bgcolor='white'
)

# Guardar en HTML
archivo_html = "data/16dic25/grafo_hvg_2020_2025.html"
print(f"Guardando visualización en {archivo_html}...")
fig.write_html(archivo_html)
print(f"✓ Visualización guardada exitosamente en: {archivo_html}")

# Mostrar la figura
fig.show()


Creando grafo NetworkX...
Grafo creado con 1485 nodos y 2959 enlaces
Calculando layout del grafo...
Creando trazas de enlaces...
Creando traza de nodos...
Creando traza de serie temporal...
Creando figura con subplots...
Guardando visualización en data/16dic25/grafo_hvg_2020_2025.html...
✓ Visualización guardada exitosamente en: data/16dic25/grafo_hvg_2020_2025.html


In [9]:
# Create graph visualization with Plotly using force layout (English version)
# Including the IMF time series in the same figure
import networkx as nx
from plotly.subplots import make_subplots

# Validate that there is data to visualize
if len(indices_filtrados) == 0:
    raise ValueError("No nodes in the specified date range. Adjust the date range.")

if len(enlaces_mapeados) == 0:
    print("⚠ Warning: No edges in the filtered subgraph.")

# Convert values to numpy arrays to ensure compatibility
valores_imf_array = np.array(valores_imf_filtrados, dtype=np.float64)

# Create NetworkX graph to calculate layout
print("Creating NetworkX graph...")
G = nx.Graph()
G.add_nodes_from(range(len(indices_filtrados)))
if len(enlaces_mapeados) > 0:
    # Ensure edges are integer tuples
    enlaces_tuplas = [(int(edge[0]), int(edge[1])) for edge in enlaces_mapeados]
    G.add_edges_from(enlaces_tuplas)
    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# Calculate layout using spring layout (force-directed)
print("Calculating graph layout...")
pos = nx.spring_layout(G, k=1, iterations=50, seed=42)

# Extract positions
x_pos = np.array([pos[i][0] for i in range(len(indices_filtrados))])
y_pos = np.array([pos[i][1] for i in range(len(indices_filtrados))])

# Prepare hover text more efficiently
def format_date(fecha):
    """Formats a date safely."""
    try:
        if isinstance(fecha, pd.Timestamp):
            return fecha.strftime('%Y-%m-%d')
        elif isinstance(fecha, str):
            return fecha
        else:
            return str(pd.Timestamp(fecha).strftime('%Y-%m-%d'))
    except:
        return str(fecha)

hover_texts = [
    f"Date: {format_date(fecha)}<br>IMF_1 Value: {val:.4f}<br>Index: {idx}"
    for fecha, val, idx in zip(fechas_filtradas, valores_imf_array, indices_filtrados)
]

# Create edge traces more efficiently
print("Creating edge traces...")
edge_x = []
edge_y = []
if len(enlaces_mapeados) > 0:
    for edge in enlaces_mapeados:
        src_idx = int(edge[0])
        dst_idx = int(edge[1])
        x0, y0 = pos[src_idx]
        x1, y1 = pos[dst_idx]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

# Create edge trace
edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    showlegend=False
)

# Calculate value range for color scale (shared between graph and time series)
valor_min = np.min(valores_imf_array)
valor_max = np.max(valores_imf_array)

# Create node trace
print("Creating node trace...")
node_trace = go.Scatter(
    x=x_pos,
    y=y_pos,
    mode='markers',
    marker=dict(
        size=8,
        color=valores_imf_array,
        colorscale='Viridis',
        cmin=valor_min,  # Minimum range for scale
        cmax=valor_max,  # Maximum range for scale
        showscale=False,  # Hide colorbar from graph, will be shown only in time series
        line=dict(width=1, color='black')
    ),
    text=hover_texts,
    hoverinfo='text',
    showlegend=False
)

# Create time series trace with colors based on values (same heatmap as graph)
print("Creating time series trace...")
time_series_trace = go.Scatter(
    x=fechas_filtradas,
    y=valores_imf_array,
    mode='lines+markers',
    name='IMF_1',
    line=dict(color='rgba(128,128,128,0.3)', width=1),  # Subtle gray line to connect points
    marker=dict(
        size=4,
        color=valores_imf_array,
        colorscale='Viridis',
        cmin=valor_min,  # Same range as graph
        cmax=valor_max,  # Same range as graph
        showscale=True,  # Only this trace will show the colorbar
        colorbar=dict(title="IMF_1 Value", x=1.02, len=0.4, y=0.25),
        line=dict(width=0.5, color='white')
    ),
    hovertemplate='Date: %{x}<br>IMF_1 Value: %{y:.4f}<extra></extra>'
)

# Create figure with subplots: graph on top, time series on bottom
print("Creating figure with subplots...")
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.6, 0.4],  # 60% for graph, 40% for time series
    vertical_spacing=0.12
)

# Add graph traces to first subplot
fig.add_trace(edge_trace, row=1, col=1)
fig.add_trace(node_trace, row=1, col=1)

# Add time series trace to second subplot
fig.add_trace(time_series_trace, row=2, col=1)

# Update first subplot layout (graph)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    row=1, col=1
)
fig.update_yaxes(
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    row=1, col=1
)

# Update second subplot layout (time series)
fig.update_xaxes(
    title_text="Date",
    row=2, col=1
)
fig.update_yaxes(
    title_text="IMF_1 Value",
    row=2, col=1
)

# Update general layout
fig.update_layout(
    showlegend=False,
    hovermode='closest',
    height=900,
    margin=dict(b=50, l=50, r=50, t=50),
    plot_bgcolor='white'
)

# Save to HTML
archivo_html = "data/16dic25/grafo_hvg_2020_2025_english.html"
print(f"Saving visualization to {archivo_html}...")
fig.write_html(archivo_html)
print(f"✓ Visualization saved successfully to: {archivo_html}")

# Display the figure
fig.show()

Creating NetworkX graph...
Graph created with 1485 nodes and 2959 edges
Calculating graph layout...
Creating edge traces...
Creating node trace...
Creating time series trace...
Creating figure with subplots...
Saving visualization to data/16dic25/grafo_hvg_2020_2025_english.html...
✓ Visualization saved successfully to: data/16dic25/grafo_hvg_2020_2025_english.html
